In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import librosa
from pathlib import Path
import torch.nn as nn
from torch.optim import Adam


In [2]:
train_protocol = Path(r"/home/achaammamathayi/Academia/Projects/Datasets/ASVSpoof19/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
dev_protocol = Path(r"/home/achaammamathayi/Academia/Projects/Datasets/ASVSpoof19/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt")
eval_protocol = Path(r"/home/achaammamathayi/Academia/Projects/Datasets/ASVSpoof19/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt")
train_path = Path(r"/home/achaammamathayi/Academia/Projects/Datasets/ASVSpoof19/LA/ASVspoof2019_LA_train/flac")
dev_path = Path(r"/home/achaammamathayi/Academia/Projects/Datasets/ASVSpoof19/LA/ASVspoof2019_LA_dev/flac")
eval_path = Path(r"/home/achaammamathayi/Academia/Projects/Datasets/ASVSpoof19/LA/ASVspoof2019_LA_eval/flac")


In [3]:
# Draft3 Dataset class
class ASVDataset(Dataset):
    def __init__(self, audio_folder, label_file, mfcc_cache = Path("mfcc_cache")):
        super().__init__()
        self.audio_folder = audio_folder
        self.label_file = label_file
        self.label_tuple = self.parser()
        self.mfcc_cache = mfcc_cache
        self.mfcc_cache.mkdir(parents=True,exist_ok=True)

# Parser function
    def parser(self):
        self.label_tuple = []
        with open(self.label_file, 'r') as f:
            for line in f:
                parts = line.split()
                file_name = parts[1] + '.flac'
                bonorpspoof = 1 if parts[4] == 'bonafide' else 0
                if self.audio_folder.joinpath(file_name).exists():
                    self.label_tuple.append([file_name,bonorpspoof])
                else:
                     print(f'Missing file: {self.audio_folder.joinpath(file_name)}')
                     continue
        return self.label_tuple

# Creating a cache folder to be in limits of running the code in Uni HPC cluster
    def mfcc_cache_gen(self,idx):
        # Plan: extract audio from the full_path. link it with the label. That should give something like mfcc sample, label.
        audio_path = self.audio_folder.joinpath(self.label_tuple[idx][0])
        filename = f"mfcc_{Path(self.label_tuple[idx][0]).stem}.npy"
        cache_path = self.mfcc_cache.joinpath(filename)
        if not cache_path.exists():
            m, sr = librosa.load(path=audio_path,sr=16000)
            mfcc = librosa.feature.mfcc(y=m, n_mfcc=20,)
            if mfcc.shape[1] > 125:
                mfcc = mfcc[:, : 125]
            elif mfcc.shape[1] == 125:
                pass
            elif mfcc.shape[1] < 125:
                pad_size = 125 - mfcc.shape[1]
                mfcc = np.pad(mfcc, ((0,0), (0, pad_size)), 'constant', constant_values=0)
            np.save(cache_path, mfcc)
        else:
            mfcc = np.load(cache_path)

        return mfcc
  
    
    def __len__(self):
        return len(self.label_tuple)
    
    def __getitem__(self, index):
        x = torch.tensor(self.mfcc_cache_gen(index), dtype=torch.float32)
        y= torch.tensor(self.label_tuple[index][1])
        return x,y

In [4]:
# DataLoader class
train_dataset = ASVDataset(train_path,train_protocol)
dev_dataset = ASVDataset(dev_path, dev_protocol)
eval_dataset = ASVDataset(eval_path, eval_protocol)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,drop_last=True)
dev_loader = DataLoader(dev_dataset, batch_size=128, shuffle=False, drop_last=False)
eval_loader = DataLoader(eval_dataset, batch_size=128, shuffle=False, drop_last=False)

Missing file: /home/achaammamathayi/Academia/Projects/Datasets/ASVSpoof19/LA/ASVspoof2019_LA_train/flac/LA_T_9667455.flac


In [5]:
# Model design
class ASVCNN(nn.Module):
    def __init__(self, input_size=1, num_classes = 1):
        super(ASVCNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=input_size, out_channels=32, kernel_size=(3,5), stride=1, padding=(1,2)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # might have to change the pooling kernel stride to an overlapping one after the training
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        )
        self.layer2 = nn.Sequential(
            #kernel size remains the same, hence the padding and striding as well
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,5), stride=1, padding=(1,2)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            #changing the pooling kernel size so that the layer 1 feautremap are made in a higher resolution feature map.
            #no pooling around mfcc axis, only pooling along time frame axis
            nn.MaxPool2d(kernel_size=(1,2), stride=(1,2))
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3,5), stride=1, padding=(1,2)),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        )
        #classification layers
        self.linear1 = nn.Sequential(
            nn.Linear(9600,128),
            nn.ReLU()
        )
        self.linear2 = nn.Linear(128, num_classes)

    def forward(self,x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        # print('layer1: ',x.shape)
        x = self.layer2(x)
        # print('layer 2',x.shape)
        x = self.layer3(x)
        # print('layer 3',x.shape)

        #flattening x
        x = torch.flatten(x, start_dim=1)
        # print('after flattening',x.shape)

        x = self.linear1(x)
        # print('linear 1:', x.shape)
        x = self.linear2(x)
        # print('linear 2:',x.shape)
        return x

In [6]:
# the training loop
def train_epoch(model, dataloader, criterion, optimizer, device):

    running_loss = 0
    model.train()

    for sample, label in dataloader:
        sample = sample.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(sample)
        label = (label.view(-1,1)).float()
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    return running_loss/len(dataloader)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ASVCNN().to(device)
dataloader = train_loader

# Calculating the total negatives/positives for pos_weight argument in BCEWithLogitsLoss()
bonfi = 0
spoof = 0
for i in train_dataset.label_tuple:
    if i[1] == 1:
        bonfi += 1
    else:
        spoof += 1
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(spoof/bonfi).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-3)

epochs = 10

for epoch in range(epochs):
    epoch_loss = train_epoch(model, dataloader, criterion, optimizer, device)
    print(f'Epoch: {epoch+1} ==================== Loss: {epoch_loss: .4f}')

# Saving the model, model weights are only saved since  saving the whole model can cause a fail
# in the future if the model code gets changed
torch.save(model.state_dict(), Path("/home/achaammamathayi/Academia/Projects/Attacker fingerprinting/trained_model.pth"))


Epoch: 1 ==================== Loss:  0.3628
Epoch: 2 ==================== Loss:  0.1097
Epoch: 3 ==================== Loss:  0.0734
Epoch: 4 ==================== Loss:  0.0555
Epoch: 5 ==================== Loss:  0.0216
Epoch: 6 ==================== Loss:  0.0398
Epoch: 7 ==================== Loss:  0.0302
Epoch: 8 ==================== Loss:  0.0155
Epoch: 9 ==================== Loss:  0.0264
Epoch: 10 ==================== Loss:  0.0357


In [7]:
# Eval loop

model = ASVCNN()
saved_weights = torch.load(Path("/home/achaammamathayi/Academia/Projects/Attacker fingerprinting/trained_model.pth"), weights_only=True)
model.load_state_dict(saved_weights)
model.to(device)

def eval_model(model, dataloader, criterion,device):

    running_loss = 0
    model.eval()

    with torch.no_grad():
        for sample, label in dataloader:
            sample = sample.to(device)
            label = label.to(device)

            output = model(sample)
            label = (label.view(-1,1)).float()
            loss = criterion(output, label)
            running_loss += loss.item()

        return running_loss/len(dataloader)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataloader = dev_loader
criterion = nn.BCEWithLogitsLoss()


epoch_loss = eval_model(model, dataloader, criterion, device)
print(f'Eval on dev set; Loss: {epoch_loss: .4f}')

    

Eval on dev set; Loss:  0.0551
